# IDA CFG Analysis with ipysigma

This notebook demonstrates how to extract, load, and visualize the Control Flow Graph (CFG) from an IDA database.


### 1. Extract CFG from IDA Database
This step requires `idapro` 9.0+ and a valid license.


In [ ]:
import sys
import os
from src.extract_cfg import extract_cfg_from_db

# Replace with the path to your IDA database (.idb or .i64)
db_path = "/Users/mark/windows_share/test/reorder_and_pad.exe.i64"
json_path = "../data/parsed_xref_graph.json"

if os.path.exists(db_path):
    json_path = extract_cfg_from_db(db_path, output_path=json_path)
else:
    print(f"Database {db_path} not found. Skipping extraction.")


### 2. Load and Visualize the Graph with ipysigma


In [2]:
#from idadex import idaapi
from typing import Hashable
from networkx import k_core, DiGraph
import importlib
importlib.invalidate_caches()
from ipysigma import Sigma
import pandas as pd
import networkx as nx
import src.visualize_cfg as cfg
importlib.reload(cfg)
import os



def prune_graph(og: DiGraph[Hashable]) -> DiGraph[Hashable]:
    to_return = og.copy()
    print(f"Graph loaded: {len(to_return.nodes)} nodes, {len(to_return.edges)} edges")
    entrypoint = [x for x in to_return.nodes() if to_return.nodes[x]['entry_point'] == True]
    print(f"entrypoint is {entrypoint}")

    # remove self loops
    to_return.remove_edges_from(nx.selfloop_edges(to_return))

    to_return = collapse_thunks(to_return)
    to_return = cfg.collapse_chains(to_return)

    # remove nodes with degree <= 1 - ALSO REMOVING ISOLATED LEAVES
    #to_return = k_core(to_return, 2)
    #entrypoint = [x for x in to_return.nodes() if to_return.nodes[x]['entry_point'] == True]
    #print(f"entrypoint is {entrypoint}")
    # to_be_removed = [x for  x in G.nodes() if G.degree()[x] <= 1]
    # print(f"Number of nodes to be removed: {len(to_be_removed)}")
    # G.remove_nodes_from(to_be_removed)
    # # Basic info
    # print(f"Number of functions after removing degree <= 1: {len(G.nodes)}")
    #
    # print(f"Number of nodes after collapsing chains: {len(G.nodes)}")
    # entrypoint = [x for x in G.nodes() if G.nodes[x]['entry_point'] == True]
    # if not entrypoint:
    #     print("No entrypoint found. Please check the graph.")
    #     raise ValueError("No entrypoint found")
    print(f"Graph pruned: {len(to_return.nodes)} nodes, {len(to_return.edges)} edges")
    return to_return

def collapse_thunks(G: nx.DiGraph):
    collapsed_G = G.copy()
    nodes_to_process = list(collapsed_G.nodes())

    for node in nodes_to_process:

        flags = collapsed_G.nodes[node].get('flags', {})
        if flags & 128:
            preds = list(collapsed_G.predecessors(node))
            succs = list(collapsed_G.successors(node))
            if len(succs) == 1:
                collapsed_G.remove_node(node)
                for pred in preds:
                    collapsed_G.add_edge(pred, succs[0])
    return collapsed_G

# Load the graph data
G = cfg.load_cfg(json_path)
if G is None:
    print(f"Could not load graph from {json_path}. Please make sure the file exists and it is a valid CFG JSON.")
    exit()

assert(G is not None)
pruned = prune_graph(G)

largest_degree = 0
def set_size(n, d) -> int:
    return len(d.get('instrs'))+G.out_degree(n)

Graph loaded: 5174 nodes, 8729 edges
entrypoint is ['0x1400155d8']
Graph pruned: 5025 nodes, 8473 edges


In [ ]:

# Visualize with ipysigma
widget = Sigma(
    pruned,
    node_label='label',
    raw_node_color=lambda n, d: "red" if "main" in d.get("func") else d.get('color'),
    edge_color='type',
    label_font='sans-serif',
    default_edge_type='arrow',
    start_layout=10,
)
print(largest_degree)
display(widget)


0


Sigma(nx.DiGraph with 5,025 nodes and 8,473 edges)

Ways to reduce:
- strongly connected components
- remove nodes with degree <= 1
- clique detection and collapse
- graph community collapse
min cuts
- graph spectral?
- laplacian?
- adamic-adar index?

In [ ]:
## needed to reload code changed on disk, this is the correct import
import jsonlines
summaries = dict()
with jsonlines.open("../data/cache_full_func.json", "r") as f:
    for line in f:

        summaries.update(line)

In [ ]:
for node in pruned.nodes():
    pruned.nodes[node]['summary'] = summaries.get(pruned.nodes[node].get('func'))


In [14]:
import math
from collections import defaultdict

# Get a position for every block within the same function and create a circular call graph visualization
def function_cluster_layout(G, radius=10, spacing=80):
    funcs = defaultdict(list)
    for n, d in G.nodes(data=True):
        funcs[d.get("func")].append(n)

    layout = {}
    func_names = sorted(funcs)

    # Get grid size depending on the number of functions
    cols = math.ceil(math.sqrt(len(func_names)))


    for i, func in enumerate(func_names):
        # Get grid position
        cx = (i % cols) * spacing
        cy = (i // cols) * spacing

        # Order blocks by their adresses (since circular, maybe not very useful, but could try linear)
        blocks = sorted(funcs[func])

        for j, node in enumerate(blocks):
            # Create circle with blocks within the same function
            angle = 2 * math.pi * j / max(1, len(blocks))
            layout[node] = {
                "x": cx + radius * math.cos(angle),
                "y": cy + radius * math.sin(angle),
            }

    return layout

# Make intra-function edges weight higher so that ForceAtlas keeps the function together when visualizing
def add_force_weights(G, intra_weight=80.0, inter_weight=0.001):
    H = G.copy()

    for u, v, d in H.edges(data=True):
        edge_type = d.get("type")

        u_funcs = set(H.nodes[u].get("funcs", [H.nodes[u].get("func")]))
        v_funcs = set(H.nodes[v].get("funcs", [H.nodes[v].get("func")]))
        same_function = bool(u_funcs & v_funcs)

        if edge_type == "intra-function" and same_function:
            d["weight"] = intra_weight
        elif edge_type in {"inter-function", "non-call", "imported-function"}:
            d["weight"] = inter_weight

    return H

In [15]:
# visualize with ipysigma, now with summaries
weighted_g = add_force_weights(pruned)

layout = function_cluster_layout(weighted_g,radius=12, spacing=120)

widget = Sigma(
    weighted_g,
    layout=layout,
    start_layout=False,
    node_label="label",
    raw_node_color=lambda n, d: d.get("color"),
    edge_color="type",
    default_edge_type="arrow",
    height=800
)
display(widget)

Sigma(nx.DiGraph with 5,025 nodes and 8,473 edges)

In [53]:
# Visualize Level0 summaries - which are summaries for each function
def get_level0_summary_graph(G, *, include_non_call=True):
    function_graph = nx.DiGraph()
    call_edge_types = {"inter-function", "imported-function"}
    if include_non_call:
        call_edge_types.add("non-call")

    # Collapse CFG block nodes into one node per function.
    for block_id, block_data in G.nodes(data=True):
        func = block_data.get("func", str(block_id))
        if func not in function_graph:
            function_graph.add_node(
                func,
                name=func,
                summary=block_data.get("summary", ""),
                color=block_data.get("color"),
                entry_point=False,
                block_ids=[],
                block_count=0,
                instruction_count=0,
                callers=[],
                callees=[],
                incoming_calls=[],
                outgoing_calls=[],
            )

        func_data = function_graph.nodes[func]
        func_data["block_ids"].append(block_id)
        func_data["block_count"] += 1
        func_data["instruction_count"] += len(block_data.get("instrs", []))
        func_data["entry_point"] = func_data["entry_point"] or block_data.get("entry_point", False)
        if not func_data.get("summary") and block_data.get("summary"):
            func_data["summary"] = block_data["summary"]

    # Keep only edges that cross function boundaries.
    for src_block, dst_block, edge_data in G.edges(data=True):
        edge_type = edge_data.get("type", "unknown")
        if edge_type not in call_edge_types:
            continue

        src_func = G.nodes[src_block].get("func", str(src_block))
        dst_func = G.nodes[dst_block].get("func", str(dst_block))
        if src_func == dst_func:
            continue

        call = {
            "src_block": src_block,
            "dst_block": dst_block,
            "src_func": src_func,
            "dst_func": dst_func,
            "type": edge_type,
            "conditional": edge_data.get("conditional", False),
        }

        if function_graph.has_edge(src_func, dst_func):
            func_edge = function_graph.edges[src_func, dst_func]
            func_edge["call_count"] += 1
            func_edge["call_sites"].append(call)
            func_edge["edge_types"] = sorted(set(func_edge["edge_types"]) | {edge_type})
            func_edge["conditional"] = func_edge["conditional"] or edge_data.get("conditional", False)
        else:
            function_graph.add_edge(
                src_func,
                dst_func,
                type=edge_type,
                edge_types=[edge_type],
                call_count=1,
                call_sites=[call],
                conditional=edge_data.get("conditional", False),
            )

    # Add caller/callee metadata after all edges have been collapsed.
    for func in function_graph.nodes():
        callers = sorted(function_graph.predecessors(func))
        callees = sorted(function_graph.successors(func))
        function_graph.nodes[func]["callers"] = callers
        function_graph.nodes[func]["callees"] = callees
        function_graph.nodes[func]["incoming_calls"] = [
            call
            for caller in callers
            for call in function_graph.edges[caller, func]["call_sites"]
        ]
        function_graph.nodes[func]["outgoing_calls"] = [
            call
            for callee in callees
            for call in function_graph.edges[func, callee]["call_sites"]
        ]

    return function_graph


level0_summary_graph = get_level0_summary_graph(pruned)

print(
    f"Function summary graph: {level0_summary_graph.number_of_nodes()} functions, "
    f"{level0_summary_graph.number_of_edges()} function-level calls"
)

# Example: inspect one function node with summary/caller/callee metadata.
next(iter(level0_summary_graph.nodes(data=True)))


Function summary graph: 335 functions, 681 function-level calls


('sub_140016F54',
 {'name': 'sub_140016F54',
  'summary': 'Function sub_140016F54:\n--- Block sub_140016F54 @ 0x140016f54 ---\nSVC             3',
  'color': '#eb500e',
  'entry_point': False,
  'block_ids': ['0x140016f54'],
  'block_count': 1,
  'instruction_count': 1,
  'callers': [],
  'callees': [],
  'incoming_calls': [],
  'outgoing_calls': []})

In [54]:
# Visualise function call graph
widget = Sigma(
    level0_summary_graph,
    start_layout=False,
    node_label="name",
    raw_node_color=lambda n, d: d.get("color"),
    edge_color="type",
    default_edge_type="arrow",
    height=800
)
display(widget)


Sigma(nx.DiGraph with 335 nodes and 681 edges)

In [ ]:
# Level1 summaries - Skip over functions that are not called/connected to anything. Maybe create a summary of those functions together that explains why they aren't connected? 
# Summarize nodes that have callees that do not have outgoing points to there parent

def get_level1_summary_graph():
    pass